## Resume Job Matching

## Linked In Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arshkon/linkedin-job-postings")

print("Path to dataset files:", path)

100%|██████████| 159M/159M [00:01<00:00, 92.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/arshkon/linkedin-job-postings/versions/13


In [ ]:
import os 
os.listdir(path)

['postings.csv', 'jobs', 'companies', 'mappings']

In [3]:
import pandas as pd
df=pd.read_csv(os.path.join(path,"postings.csv"))

In [4]:
df.head(3)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [6]:
df=df[["title","description","skills_desc"]]

## Handle missing val

In [7]:
df=df.dropna()

In [8]:
df.shape

(2439, 3)

In [9]:
df = df.reset_index(drop=True)

In [10]:
df.head(5)

,title,description,skills_desc
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...
1,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...
2,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,This position requires a baseline understandin...
3,Respiratory Therapist,"At Children’s, the region’s only full-service ...",• Requires the ability to communicate effectiv...
4,Worship Leader,It is an exciting time to be a part of our chu...,"Knowledge, Skills and Abilities: 1. Proficient..."


## combining and preprocesing text

In [11]:
import re

def preprocessed_text(row):
    text=[]
    for col in df.columns:
        text.append(f"{col}: {str(row[col])}")


    text=" ".join(text)
    text=text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text


df["combined_features"]=df.apply(preprocessed_text,axis=1)


In [12]:
df.head()

,title,description,skills_desc,combined_features
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...,title marketing coordinator description job de...
1,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...,title assitant restaurant manager description ...
2,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,This position requires a baseline understandin...,title senior elder law trusts and estates ass...
3,Respiratory Therapist,"At Children’s, the region’s only full-service ...",• Requires the ability to communicate effectiv...,title respiratory therapist description at chi...
4,Worship Leader,It is an exciting time to be a part of our chu...,"Knowledge, Skills and Abilities: 1. Proficient...",title worship leader description it is an exci...


## Genrate Embedding

In [ ]:
sentences=df["combined_features"].apply(lambda X:X.split()).tolist()
sentences

In [15]:
import gensim
from gensim.models import Word2Vec

model=Word2Vec(sentences,vector_size=200,window=10,epochs=15,min_count=1, sg=1)

In [16]:
import pickle
with open("model.pkl","wb") as f:
  pickle.dump(model,f)
  f.close()

In [17]:
import numpy as np

def avg_w2v(text,model):
    words=text.split()
    words_vector=[model.wv[word] for word in words if word in model.wv]
    return np.mean(words_vector,axis=0) if words_vector else np.zeros(model.vector_size)

df["embeddings"]=df["combined_features"].apply(lambda text:avg_w2v(text,model))

In [18]:
df["embeddings"][1].shape
df.info()
df.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2439 entries, 0 to 2438
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              2439 non-null   object
 1   description        2439 non-null   object
 2   skills_desc        2439 non-null   object
 3   combined_features  2439 non-null   object
 4   embeddings         2439 non-null   object
dtypes: object(5)
memory usage: 95.4+ KB


,title,description,skills_desc,combined_features,embeddings
0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,Requirements: \n\nWe are seeking a College or ...,title marketing coordinator description job de...,"[-0.006503554, -0.07292572, 0.072199725, 0.189..."
1,Assitant Restaurant Manager,The National Exemplar is accepting application...,We are currently accepting resumes for FOH - A...,title assitant restaurant manager description ...,"[-0.05207241, -0.08931895, 0.03871853, 0.21556..."


In [19]:
import pickle
with open("dataset.pkl","wb") as f:
  pickle.dump(df,f)
  f.close()

## Reccomendation

In [87]:
import pickle
from sklearn.metrics.pairwise import cosine_similarity

with open("dataset.pkl","rb")as f:
  dataset=pickle.load(f)

with open("model.pkl","rb")as f:
  model=pickle.load(f)


def get_recomendation(resume):
  resume=resume.lower()
  resume=re.sub(r"[^a-z0-9\s]", "", resume)
  words=resume.split()
  word_vec=[model.wv[word] for word in words if word in model.wv]
  final_res=np.mean(word_vec,axis=0) if word_vec else np.zeros(model.vector_size)

  recomendation=[]

  for idx,emb in enumerate(dataset["embeddings"]):
    recomendation.append((cosine_similarity([final_res], [emb])[0][0],int(idx)))

  recomendation.sort(reverse=True,key=lambda x:x[0])
  seen_titles=set()
  results=[]

  for it in recomendation[:100]:
    title=dataset.iloc[it[1]]["title"]
    score=it[0]
    if len(seen_titles)==5:
      break
    if title not in seen_titles and score>=.80:
      seen_titles.add(title)
      results.append((title,score))

  return results

resultant=get_recomendation(resume)
if not resultant:
    resultant.append(("nothing can be done sorry",00))

for it in resultant:
  print(f"Job Title | {it[0]} | Similarity score = { it[1] }")


Job Title | Analyst | Similarity score = 0.9612547755241394
Job Title | Data Engineer | Similarity score = 0.9603559374809265
Job Title | Volunteer: Database Architect | Similarity score = 0.9571632146835327
Job Title | Palantir Developer | Similarity score = 0.9533262848854065
Job Title | Manager, Data Analytics | Similarity score = 0.9532912969589233


In [86]:
resume="""John Doe
Email: john.doe@example.com | Phone: (123) 456-7890
LinkedIn: linkedin.com/in/johndoe | GitHub: github.com/johndoe

Objective:
A detail-oriented and results-driven Data Scientist with 3+ years of experience in machine learning, data analysis, and statistical modeling. Passionate about applying data-driven solutions to real-world problems and contributing to business growth. Looking to leverage my skills in data science and machine learning to drive innovative solutions.

Skills:

    Programming Languages: Python, R, SQL, JavaScript
    Libraries/Frameworks: Pandas, NumPy, Scikit-learn, TensorFlow, Keras, PyTorch
    Machine Learning: Regression, Classification, Clustering, Time Series Analysis
    Data Visualization: Matplotlib, Seaborn, Plotly, Tableau
    Databases: MySQL, MongoDB, PostgreSQL
    Tools: Jupyter Notebook, Git, Docker, AWS, Google Cloud
    Operating Systems: Linux, Windows, macOS
    Soft Skills: Problem Solving, Communication, Teamwork, Time Management

Professional Experience:

Data Scientist | XYZ Corp
May 2021 – Present

    Developed machine learning models to predict customer churn, improving retention rates by 20%.
    Built recommendation systems using collaborative filtering and content-based methods.
    Designed data pipelines to automate the collection, cleaning, and transformation of large datasets.
    Performed exploratory data analysis (EDA) to identify trends, correlations, and patterns in sales data.
    Collaborated with business stakeholders to define project requirements and deliver actionable insights.

Data Analyst | ABC Solutions
June 2019 – April 2021

    Analyzed customer data using SQL and Python to generate reports on product performance.
    Conducted A/B testing to measure the impact of marketing campaigns on sales and engagement.
    Developed dashboards in Tableau to visualize key performance indicators (KPIs) for executive leadership.
    Worked with cross-functional teams to optimize data workflows and improve operational efficiency.

Education:

Bachelor of Science in Computer Science
University of California, Berkeley
Graduated: May 2019

    Relevant Coursework: Data Structures, Algorithms, Machine Learning, Data Mining, Database Systems
    Capstone Project: Built a machine learning model to predict housing prices using regression techniques.

Certifications:

    Machine Learning by Stanford University (Coursera)
    Data Science Professional Certificate by IBM (Coursera)
    Deep Learning Specialization by Andrew Ng (Coursera)

Projects:

Predicting Customer Churn (Data Scientist, XYZ Corp)

    Developed a machine learning model using logistic regression and random forests to predict customer churn, achieving an accuracy of 87%.
    Employed feature engineering techniques and cross-validation to fine-tune the model for better accuracy.

Stock Price Prediction (Personal Project)

    Built a predictive model for stock price movement using time series analysis and LSTM networks.
    Utilized historical data to train the model and tested its performance on unseen data.

Publications:

    “Applying Machine Learning to Real-World Problems in Business,” Journal of Data Science (2020)"""